In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/mandi_prices_raw.csv")
up_df = df[df['STATE'] == 'Uttar Pradesh'].copy()

In [3]:
up_df.isnull().sum()

STATE            0
District Name    0
Market Name      0
Commodity        0
Variety          0
Grade            0
Min_Price        0
Max_Price        0
Modal_Price      0
Price Date       0
dtype: int64


Section 1-3: No missing values found in any column (0 across all 10 columns). No drop/fill decisions required. Note: zero-price values will still be checked separately in the outlier section, since 0 ≠ NaN.


In [4]:
up_df['Price Date'].head(10)

2     6/6/2023
7     6/6/2023
10    6/6/2023
12    6/6/2023
15    6/6/2023
30    6/6/2023
32    6/6/2023
46    6/6/2023
48    6/6/2023
49    6/6/2023
Name: Price Date, dtype: str

In [5]:
up_df['Price Date'] = pd.to_datetime(up_df['Price Date'], format='%d/%m/%Y')

ValueError: time data "6/13/2023" doesn't match format "%d/%m/%Y". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [6]:
# Split the date string to look at the first number (potential day or month)
first_parts = up_df['Price Date'].str.split('/').str[0].astype(int)
(first_parts > 12).sum()

np.int64(0)

In [7]:
up_df['Price Date'] = pd.to_datetime(up_df['Price Date'], format='%m/%d/%Y')

In [8]:
up_df.dtypes
up_df['Price Date'].min(), up_df['Price Date'].max()

(Timestamp('2023-06-06 00:00:00'), Timestamp('2025-06-11 00:00:00'))

Initial assumption (based on Indian data conventions) was that Price Date was D/M/YYYY. Conversion failed with error on value '6/13/2023' — '13' cannot be a month, proving D/M/YYYY was wrong for at least this value. Tested hypothesis that the entire column is M/D/YYYY by checking if any 'day' position value exceeds 12 (first_parts > 12) — result was 0, strongly supporting M/D/YYYY for the whole column (likely due to US-locale processing during the Kaggle dataset's creation). Converted successfully using format='%m/%d/%Y'.

In [9]:
up_df.dtypes

STATE                       str
District Name               str
Market Name                 str
Commodity                   str
Variety                     str
Grade                       str
Min_Price               float64
Max_Price               float64
Modal_Price             float64
Price Date       datetime64[us]
dtype: object

In [10]:
up_df.duplicated().sum()

np.int64(0)

Section 6-7: Checked for exact duplicate rows using duplicated().sum() — result was 0. No duplicate removal needed

In [11]:
up_df.groupby('Commodity')['Modal_Price'].describe()

,count,mean,std,min,25%,50%,75%,max
Commodity,,,,,,,,
Onion,70476.0,2214.634230,944.808945,460.0,1520.0,1900.0,2850.0,8700.0
Potato,85744.0,1357.411947,519.706162,300.0,980.0,1190.0,1770.0,7420.0
Rice,4196.0,3229.150143,393.682862,1350.0,3140.0,3240.0,3330.0,8470.0
Tomato,7695.0,3724.569591,2729.965378,300.0,1250.0,2350.0,6650.0,10250.0
Wheat,19983.0,2353.484212,130.661172,230.0,2271.0,2330.0,2445.0,7040.0


In [12]:
(up_df[['Min_Price', 'Max_Price', 'Modal_Price']] <= 0).sum()

Min_Price        5
Max_Price      450
Modal_Price      0
dtype: int64

In [13]:
up_df[up_df['Max_Price'] <= 0].head(10)

,STATE,District Name,Market Name,Commodity,Variety,Grade,Min_Price,Max_Price,Modal_Price,Price Date
961,Uttar Pradesh,sonbhadra,Dudhi,Onion,Onion,FAQ,800.0,0.0,900.0,2023-06-06
1414,Uttar Pradesh,sonbhadra,Dudhi,Potato,Other,FAQ,900.0,0.0,1000.0,2023-06-06
1821,Uttar Pradesh,sonbhadra,Dudhi,Tomato,Deshi,FAQ,900.0,0.0,1000.0,2023-06-06
1913,Uttar Pradesh,sonbhadra,Dudhi,Tomato,Deshi,FAQ,1000.0,0.0,1100.0,2023-06-07
3068,Uttar Pradesh,sonbhadra,Dudhi,Onion,Onion,FAQ,700.0,0.0,800.0,2023-06-07
3825,Uttar Pradesh,sonbhadra,Dudhi,Onion,Onion,FAQ,900.0,0.0,1000.0,2023-06-08
3861,Uttar Pradesh,sonbhadra,Dudhi,Potato,Other,FAQ,900.0,0.0,1000.0,2023-06-08
5093,Uttar Pradesh,sonbhadra,Dudhi,Tomato,Deshi,FAQ,900.0,0.0,1000.0,2023-06-08
5618,Uttar Pradesh,sonbhadra,Dudhi,Tomato,Deshi,FAQ,1000.0,0.0,1100.0,2023-06-09
6166,Uttar Pradesh,sonbhadra,Dudhi,Potato,Other,FAQ,1000.0,0.0,1100.0,2023-06-09


In [14]:
up_df[up_df['Max_Price'] <= 0]['Market Name'].value_counts()

Market Name
Dudhi           439
Shahaswan         2
Sikanderabad      2
Barabanki         1
Firozabad         1
Bijnaur           1
Bindki            1
Hasanpur          1
Khurja            1
Gadaura           1
Name: count, dtype: int64

In [15]:
up_df.loc[up_df['Market Name'] == 'Dudhi', 'Max_Price'] = up_df.loc[up_df['Market Name'] == 'Dudhi', 'Max_Price'].replace(0, pd.NA)

TypeError: Invalid value '[<NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA>
 <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA>
 <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA>
 <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA>
 <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA>
 <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA>
 <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA>
 <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> 1100.0 <NA>
 <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA>
 <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA>
 <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA>
 <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA>
 <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA>
 <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA

In [16]:
import numpy as np

up_df.loc[up_df['Market Name'] == 'Dudhi', 'Max_Price'] = up_df.loc[up_df['Market Name'] == 'Dudhi', 'Max_Price'].replace(0, np.nan)

In [17]:
(up_df['Max_Price'] <= 0).sum()
up_df['Max_Price'].isnull().sum()


np.int64(439)

In [18]:
print((up_df['Max_Price'] <= 0).sum())

11


In [19]:
scattered_problem = (up_df['Max_Price'] <= 0) & (up_df['Market Name'] != 'Dudhi')
up_df = up_df[~scattered_problem]

In [20]:
print(up_df.shape)
print((up_df['Max_Price'] <= 0).sum())

(188083, 10)
0


In [21]:
up_df[up_df['Min_Price'] <= 0]

,STATE,District Name,Market Name,Commodity,Variety,Grade,Min_Price,Max_Price,Modal_Price,Price Date
85819,Uttar Pradesh,sonbhadra,Dudhi,Onion,Onion,FAQ,0.0,1100.0,1000.0,2023-07-29
207909,Uttar Pradesh,basti,Basti,Potato,Desi,FAQ,0.0,1160.0,1110.0,2023-11-09
402081,Uttar Pradesh,amroha,Hasanpur,Potato,Badshah,FAQ,0.0,1010.0,950.0,2024-06-28


In [22]:
up_df = up_df[up_df['Min_Price'] > 0]

In [23]:
print(up_df.shape)
print((up_df['Min_Price'] <= 0).sum())

(188080, 10)
0


In [24]:
(up_df['Max_Price'] < up_df['Min_Price']).sum()

np.int64(0)

Examined price distributions per crop using describe() — confirmed Tomato has by far the highest volatility (std ≈ 2730 vs mean ≈ 3724), consistent with known real-world price spike behavior; decided NOT to remove high-end price values, since these represent genuine market events valuable for prediction. Found 450 rows with Max_Price ≤ 0 — investigation revealed 439 were concentrated in a single market (Dudhi, Sonbhadra), suggesting a systematic reporting gap rather than random error; replaced these with NaN (np.nan) to preserve valid Min/Modal price data rather than dropping the rows. Remaining 11 scattered Max_Price errors (across 9 different markets) were dropped as isolated entry errors. Found 3 rows with Min_Price ≤ 0, scattered across different markets with no systematic pattern — dropped these. Verified no rows violate Max_Price < Min_Price logical consistency. Final row count: 188,080 (from original 188,094 UP subset — 14 rows dropped, 439 cells set to NaN).

In [25]:
print(up_df['Commodity'].unique())
print(up_df['Variety'].unique())

<StringArray>
['Potato', 'Onion', 'Tomato', 'Wheat', 'Rice']
Length: 5, dtype: str
<StringArray>
[                    'Local',                     'Other',
                     'Deshi',                      'Dara',
                       'Red',                   'Badshah',
                    'Hybrid',                      'Desi',
                    'Tomato',               'Kufri Megha',
                    'F.A.Q.',                    'Potato',
                     'Nasik',                     'Onion',
               '147 Average',                     'Chips',
                  'Pusa-Red',                    'Medium',
              'Chandermukhi',             '(Red Nanital)',
             'Bombay (U.P.)',                  '1st Sort',
                       '343',                     'White',
                       'III',                    'Common',
                    'Coarse',                      'Fine',
     'Motta (Coarse) Boiled',        'Basmati U.P. (New)',
 'Basmati Haryana 

In [26]:
up_df['Variety'] = up_df['Variety'].str.strip().str.title()

In [27]:
clean_df = up_df.copy()
print(clean_df.shape)

(188080, 10)


In [28]:
print(clean_df.isnull().sum())
print(clean_df.duplicated().sum())
print((clean_df[['Min_Price','Max_Price','Modal_Price']] <= 0).sum())
print((clean_df['Max_Price'] < clean_df['Min_Price']).sum())
print(clean_df.dtypes)

STATE              0
District Name      0
Market Name        0
Commodity          0
Variety            0
Grade              0
Min_Price          0
Max_Price        439
Modal_Price        0
Price Date         0
dtype: int64
0
Min_Price      0
Max_Price      0
Modal_Price    0
dtype: int64
0
STATE                       str
District Name               str
Market Name                 str
Commodity                   str
Variety                     str
Grade                       str
Min_Price               float64
Max_Price               float64
Modal_Price             float64
Price Date       datetime64[us]
dtype: object


In [29]:
clean_df.to_csv("../data/processed/clean_mandi_prices.csv", index=False)


OSError: Cannot save file into a non-existent directory: '..\data\processed'

In [30]:
import os
print(os.listdir(".."))
print(os.listdir("../data"))

['.git', '.gitignore', '.ipynb_checkpoints', 'data', 'notebooks', 'requirements.txt', 'src', 'venv']
['.ipynb_checkpoints', 'raw']


In [31]:
os.makedirs("../data/processed", exist_ok=True)

In [32]:
clean_df.to_csv("../data/processed/clean_mandi_prices.csv", index=False)

In [33]:
check_df = pd.read_csv("../data/processed/clean_mandi_prices.csv")
print(check_df.shape)
print(check_df.dtypes)
print(check_df['Max_Price'].isnull().sum())

(188080, 10)
STATE                str
District Name        str
Market Name          str
Commodity            str
Variety              str
Grade                str
Min_Price        float64
Max_Price        float64
Modal_Price      float64
Price Date           str
dtype: object
439


## Day 3 Summary — Data Cleaning

**Starting point:** 188,094 rows (Uttar Pradesh subset of national mandi price dataset)

**Missing values:** None found across any column (0 nulls in original UP data).

**Date conversion:** `Price Date` was stored as text. Initial assumption (D/M/YYYY, Indian convention) failed — error revealed value `"6/13/2023"`, which is impossible under D/M/YYYY (no month 13). Tested hypothesis that the entire column was M/D/YYYY (US convention) by checking whether any "first number" exceeded 12 — result was 0 occurrences, strongly supporting M/D/YYYY for the whole column. Converted successfully using `format='%m/%d/%Y'`. Final date range: 2023-06-06 to 2025-06-11 (~2 years).

**Duplicates:** Checked using `duplicated().sum()` — result was 0. No duplicate rows existed.

**Price validity (Max_Price):** Found 450 rows with `Max_Price <= 0`. Investigation revealed 439 of these (97.6%) were concentrated in a single market — Dudhi, Sonbhadra district — across multiple dates and commodities, with Min_Price and Modal_Price both valid for these same rows. This pattern suggested a systematic reporting gap for Max_Price specific to that market, not random error. Decision: replaced Max_Price with NaN for these 439 rows (preserving valid Min/Modal price data) rather than dropping the rows entirely. The remaining 11 scattered Max_Price errors (spread across 9 different markets, no common pattern) were treated as isolated entry errors and dropped.

**Price validity (Min_Price):** Found 3 rows with `Min_Price <= 0` (after the above fix), scattered across 3 different markets with no systematic pattern. Dropped as isolated entry errors.

**Logical consistency (Max < Min):** Checked for rows where Max_Price < Min_Price — result was 0 violations.

**Outlier philosophy:** Did NOT remove high-end price values for any crop. Examined per-crop distributions using `describe()` — confirmed Tomato shows the highest volatility (std ≈ 2730 relative to mean ≈ 3724), consistent with known real-world price spike behavior for perishable vegetables. These extreme-but-real values are valuable signal for price prediction, not noise to be removed.

**Text standardization:** Checked `Commodity` (clean, 5 crops, no spelling variants) and `Variety` (31 unique values, some unusual entries like "F.A.Q." and crop names appearing as variety values — left as-is since Variety isn't used in core project logic). Applied `.str.strip().str.title()` to Variety as a light-touch safety measure against whitespace/casing issues.

**Final row count:** 188,080 (14 rows dropped total: 11 scattered Max_Price errors + 3 Min_Price errors; 439 cells set to NaN, not dropped).

**Output:** Saved to `data/processed/clean_mandi_prices.csv`.

**Known limitation for future days:** CSV format does not preserve datetime dtype. `Price Date` will reload as plain text (`object`/`str`) in any future notebook — must re-run `pd.to_datetime(df['Price Date'], format='%m/%d/%Y')` after loading this file before doing any date-based operations (extracting month/year, sorting chronologically, computing lag features).